## DETAILED ANALYSIS OF RESULTS

Imports:

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from scipy.stats import t
import h5py

BASE_DIR = Path("/home/projects/avera/medper/uvigo_voice/metrics/metrics_paper/all_tasks")

RESULT_FILES = [
    "AA_LR_acoustic_wav2vec_wavlm.csv",
    "AA_LR_acoustic.csv",
    "BA_LR_acoustic_wav2vec_wavlm.csv",
    "BA_LR_acoustic.csv",
    "AC_LR_acoustic_wav2vec_wavlm.csv",
    "AC_LR_acoustic.csv",
    "BC_LR_acoustic_wav2vec_wavlm.csv",
    "BC_LR_acoustic.csv",
]

METRIC_COLUMNS = [
    # "accuracy_average_prob",
    "f1_score_average_prob",
    # "recall_average_prob",
    # "precision_average_prob",
]

N_FOLDS = 5
CONFIDENCE_LEVEL = 0.95
ALPHA = 1 - CONFIDENCE_LEVEL

FEATURES_DIR = Path("/home/projects/avera/medper/uvigo_voice/features/dicoperia_features")

LABELS_PATH = Path(
    "/home/projects/avera/medper/uvigo_voice/data/dicoperia/condition_dicoperia.csv"
)

ACOUSTIC_FEATURE_FILES = [
    "compare_2016_energy.h5",
    "compare_2016_voicing.h5",
    "compare_2016_rasta.h5",
    "compare_2016_basic_spectral.h5",
    "spafe_mfcc.h5",
]

SSL_FEATURE_FILES = [
    "ssl_wavlm_pre_classifier.h5",
    "ssl_xlsr_300m_utterance.h5",
]

# Model-related constants for Section 6
MODELS_BASE_DIR = Path("/home/temporal2/avera/coperia_raw/models_paper")

TASK_MODEL_DIRS = {
    "AA": MODELS_BASE_DIR / "after_a",
    "BA": MODELS_BASE_DIR / "before_a",
    "AC": MODELS_BASE_DIR / "after_cough",
    "BC": MODELS_BASE_DIR / "before_cough",
}

MODEL_CONFIGS = {
    "LR_acoustic": "aggregated_compare_2016_energy_compare_2016_voicing_compare_2016_rasta_compare_2016_basic_spectral_spafe_mfcc_LogisticRegression_*.pkl",
    "LR_acoustic_wav2vec_wavlm": "aggregated_compare_2016_energy_compare_2016_voicing_compare_2016_rasta_compare_2016_basic_spectral_spafe_mfcc_ssl_wavlm_ssl_wav2vec_LogisticRegression_*.pkl",
}

print("All imports and constants defined successfully.")

All imports and constants defined successfully.


Helper functions:

In [2]:
def compute_mean_std_ci(values: np.ndarray) -> dict:
    """
    Compute mean, sample STD, standard error, and 95% CI using Student's t distribution.
    """
    values = np.asarray(values, dtype=float)
    n = len(values)

    mean = values.mean()
    std = values.std(ddof=1)
    se = std / np.sqrt(n)

    t_crit = t.ppf(1 - ALPHA / 2, df=n - 1)
    ci_lower = mean - t_crit * se
    ci_upper = mean + t_crit * se

    return {
        "n_folds": n,
        "mean": mean,
        "std": std,
        "ci_lower": ci_lower,
        "ci_upper": ci_upper,
        "ci_half_width": t_crit * se,
    }


## Section 1 — Confidence intervals for reported CV metrics

Compute 95% confidence intervals across the five cross-validation folds for the average-probability metrics reported in the paper.

In [3]:
rows = []

for filename in RESULT_FILES:
    path = BASE_DIR / filename

    df = pd.read_csv(path, sep=";", decimal=",")

    if len(df) != N_FOLDS:
        print(f"Warning: {filename} has {len(df)} rows instead of {N_FOLDS}")

    configuration = filename.replace(".csv", "")
    
    for metric in METRIC_COLUMNS:
        stats = compute_mean_std_ci(df[metric].values)

        rows.append({
            "configuration": configuration,
            "metric": metric,
            **stats,
        })

ci_results = pd.DataFrame(rows)

ci_results["mean_std"] = ci_results.apply(
    lambda row: f"{row['mean']:.3f} ± {row['std']:.3f}",
    axis=1,
)

ci_results["mean_95ci"] = ci_results.apply(
    lambda row: f"{row['mean']:.3f} [{row['ci_lower']:.3f}, {row['ci_upper']:.3f}]",
    axis=1,
)

ci_results = ci_results[
    [
        "configuration",
        "metric",
        "mean",
        "std",
        "ci_lower",
        "ci_upper",
        "mean_std",
        "mean_95ci",
    ]
]

ci_results = ci_results.sort_values("mean", ascending=False)

ci_results

,configuration,metric,mean,std,ci_lower,ci_upper,mean_std,mean_95ci
0,AA_LR_acoustic_wav2vec_wavlm,f1_score_average_prob,0.821732,0.047414,0.762859,0.880604,0.822 ± 0.047,"0.822 [0.763, 0.881]"
4,AC_LR_acoustic_wav2vec_wavlm,f1_score_average_prob,0.808053,0.048334,0.748039,0.868068,0.808 ± 0.048,"0.808 [0.748, 0.868]"
2,BA_LR_acoustic_wav2vec_wavlm,f1_score_average_prob,0.783213,0.127048,0.625462,0.940964,0.783 ± 0.127,"0.783 [0.625, 0.941]"
5,AC_LR_acoustic,f1_score_average_prob,0.770751,0.060646,0.695449,0.846052,0.771 ± 0.061,"0.771 [0.695, 0.846]"
6,BC_LR_acoustic_wav2vec_wavlm,f1_score_average_prob,0.740981,0.099975,0.616845,0.865117,0.741 ± 0.100,"0.741 [0.617, 0.865]"
1,AA_LR_acoustic,f1_score_average_prob,0.706841,0.055855,0.637488,0.776195,0.707 ± 0.056,"0.707 [0.637, 0.776]"
7,BC_LR_acoustic,f1_score_average_prob,0.694673,0.108143,0.560396,0.828949,0.695 ± 0.108,"0.695 [0.560, 0.829]"
3,BA_LR_acoustic,f1_score_average_prob,0.666607,0.046319,0.609095,0.724120,0.667 ± 0.046,"0.667 [0.609, 0.724]"


## Section 2 - Mann-Whitney U Test

In [4]:
import sys
import copy
import yaml
import torch
import numpy as np
import pandas as pd

from pathlib import Path
from scipy import stats
from statsmodels.stats.multitest import multipletests

sys.path.append("/home/projects/avera/medper/uvigo_voice")
sys.path.append("/home/projects/avera/medper/uvigo_voice/src")

from src.common_classification import (
    extract_features,
    get_data_to_split,
)

from multiclass_classification import (
    get_labels_dict,
    get_features_and_labels,
)


CONFIG_PATH = Path("/home/projects/avera/medper/uvigo_voice/multiclass_classification_conf.yaml")

ACOUSTIC_FEATURE_TYPES = [
    "compare_2016_energy",
    "compare_2016_voicing",
    "compare_2016_rasta",
    "compare_2016_basic_spectral",
    "spafe_mfcc",
]

TASK_MOMENT_FILES = {
    "AA": "/home/projects/avera/medper/uvigo_voice/data/dicoperia/after_a.csv",
    "BA": "/home/projects/avera/medper/uvigo_voice/data/dicoperia/before_a.csv",
    "AC": "/home/projects/avera/medper/uvigo_voice/data/dicoperia/after_cough.csv",
    "BC": "/home/projects/avera/medper/uvigo_voice/data/dicoperia/before_cough.csv",
}


with open(CONFIG_PATH, "r") as f:
    config = yaml.safe_load(f)

base_model_conf = copy.deepcopy(config["model_extract_train_test"])
base_audio_conf = copy.deepcopy(config["audioprocessor_data"])

if base_model_conf["num_cores"] == "None":
    base_model_conf["num_cores"] = None

base_model_conf["wav_folder"] = "/home/temporal2/avera/coperia_raw/train/"
base_model_conf["path_extracted_features"] = "/home/projects/avera/medper/uvigo_voice/features/dicoperia_features/"
base_model_conf["load_extracted_features"] = True
base_model_conf["normalize"] = False
base_model_conf["use_lasso_selection"] = False


def build_pipeline_matrix(
    transcript_file,
    feature_type,
    use_acoustic=True,
    use_wavlm=False,
    use_wav2vec=False,
):
    model_conf = copy.deepcopy(base_model_conf)
    audio_conf = copy.deepcopy(base_audio_conf)

    audio_conf["feature_type"] = feature_type

    model_conf["global_transcript_file"] = transcript_file
    model_conf["use_acoustic_feat"] = use_acoustic
    model_conf["use_ssl_wavlm"] = use_wavlm
    model_conf["use_ssl_wav2vec"] = use_wav2vec
    model_conf["use_ssl_hubert"] = False
    model_conf["use_paralinguistic_feat"] = False
    model_conf["use_linguistic_feat"] = False

    _, labels_dict = extract_features(
        audioprocessor_data=audio_conf,
        model_conf=model_conf,
    )

    features_id, _ = get_data_to_split(labels=labels_dict)
    features_id = np.asarray(features_id)

    selected_labels_dict, keys_labels_list, canonical_audio_order = get_labels_dict(
        labels_dict=labels_dict,
        id_index=np.arange(len(features_id)),
        features_id=features_id,
        shuffle=False,
    )

    X, y, audio_labels = get_features_and_labels(
        labels_dict=selected_labels_dict,
        keys_labels_list=keys_labels_list,
        which_feature="aggregated",
        feature_type=feature_type,
        model="LogisticRegression",
        model_conf=model_conf,
        canonical_audio_order=canonical_audio_order,
        init_model=False,
    )

    if isinstance(X, torch.Tensor):
        X = X.detach().cpu().numpy()

    if isinstance(y, torch.Tensor):
        y = y.detach().cpu().numpy()

    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=int)
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

    return X, y, audio_labels, canonical_audio_order


def mann_whitney_feature_test(X, y, task_moment, representation):
    rows = []

    for feature_idx in range(X.shape[1]):
        group_control = X[y == 0, feature_idx]
        group_pasc = X[y == 1, feature_idx]

        if np.all(group_control == group_control[0]) and np.all(group_pasc == group_pasc[0]):
            u_stat = np.nan
            p_value = 1.0
            rank_biserial = 0.0
        else:
            u_stat, p_value = stats.mannwhitneyu(
                group_control,
                group_pasc,
                alternative="two-sided",
            )

            n0 = len(group_control)
            n1 = len(group_pasc)
            rank_biserial = (2 * u_stat) / (n0 * n1) - 1

        rows.append({
            "task_moment": task_moment,
            "representation": representation,
            "feature_index": feature_idx,
            "u_statistic": u_stat,
            "p_value_raw": p_value,
            "rank_biserial_effect_size": rank_biserial,
            "control_median": np.median(group_control),
            "pasc_median": np.median(group_pasc),
            "control_mean": np.mean(group_control),
            "pasc_mean": np.mean(group_pasc),
        })

    results = pd.DataFrame(rows)

    # FDR correction
    reject_fdr, p_fdr, _, _ = multipletests(
        results["p_value_raw"],
        alpha=0.05,
        method="fdr_bh",
    )

    # Bonferroni correction
    reject_bonf, p_bonf, _, _ = multipletests(
        results["p_value_raw"],
        alpha=0.05,
        method="bonferroni",
    )

    results["p_value_fdr"] = p_fdr
    results["p_value_bonferroni"] = p_bonf

    results["significant_raw"] = results["p_value_raw"] < 0.05
    results["significant_fdr"] = reject_fdr
    results["significant_bonferroni"] = reject_bonf

    return results.sort_values("p_value_raw").reset_index(drop=True)


all_mw_results = []
feature_dimensions = []

for task_moment, transcript_file in TASK_MOMENT_FILES.items():
    print("\n==============================")
    print(f"Task-moment: {task_moment}")
    print(f"Transcript file: {transcript_file}")
    print("==============================")

    X_acoustic, y_acoustic, _, order_acoustic = build_pipeline_matrix(
        transcript_file=transcript_file,
        feature_type=ACOUSTIC_FEATURE_TYPES,
        use_acoustic=True,
        use_wavlm=False,
        use_wav2vec=False,
    )

    X_fusion, y_fusion, _, order_fusion = build_pipeline_matrix(
        transcript_file=transcript_file,
        feature_type=ACOUSTIC_FEATURE_TYPES,
        use_acoustic=True,
        use_wavlm=True,
        use_wav2vec=True,
    )

    assert np.array_equal(y_acoustic, y_fusion)

    print("Acoustic feature matrix:", X_acoustic.shape)
    print("Fusion feature matrix:", X_fusion.shape)
    print("Acoustic audio slots:", len(order_acoustic))
    print("Fusion audio slots:", len(order_fusion))

    feature_dimensions.extend([
        {
            "task_moment": task_moment,
            "representation": "acoustic",
            "n_subjects": X_acoustic.shape[0],
            "n_features": X_acoustic.shape[1],
            "n_audio_slots": len(order_acoustic),
        },
        {
            "task_moment": task_moment,
            "representation": "acoustic_wavlm_wav2vec",
            "n_subjects": X_fusion.shape[0],
            "n_features": X_fusion.shape[1],
            "n_audio_slots": len(order_fusion),
        },
    ])

    mw_acoustic = mann_whitney_feature_test(
        X=X_acoustic,
        y=y_acoustic,
        task_moment=task_moment,
        representation="acoustic",
    )

    mw_fusion = mann_whitney_feature_test(
        X=X_fusion,
        y=y_fusion,
        task_moment=task_moment,
        representation="acoustic_wavlm_wav2vec",
    )

    all_mw_results.extend([mw_acoustic, mw_fusion])


mw_results = pd.concat(all_mw_results, ignore_index=True)
feature_dimensions = pd.DataFrame(feature_dimensions)

mw_summary = (
    mw_results
    .groupby(["task_moment", "representation"])
    .agg(
        n_features=("feature_index", "count"),
        n_raw_p_lt_005=("significant_raw", "sum"),
        n_fdr_p_lt_005=("significant_fdr", "sum"),
        n_bonferroni_p_lt_005=("significant_bonferroni", "sum"),
        min_raw_p=("p_value_raw", "min"),
        min_fdr_p=("p_value_fdr", "min"),
        min_bonferroni_p=("p_value_bonferroni", "min"),
        median_abs_effect_size=("rank_biserial_effect_size", lambda x: np.median(np.abs(x))),
        max_abs_effect_size=("rank_biserial_effect_size", lambda x: np.max(np.abs(x))),
    )
    .reset_index()
)

display(feature_dimensions)
display(mw_summary)


Task-moment: AA
Transcript file: /home/projects/avera/medper/uvigo_voice/data/dicoperia/after_a.csv
Acoustic feature matrix: (154, 6336)
Fusion feature matrix: (154, 11712)
Acoustic audio slots: 3
Fusion audio slots: 3

Task-moment: BA
Transcript file: /home/projects/avera/medper/uvigo_voice/data/dicoperia/before_a.csv
Acoustic feature matrix: (154, 6336)
Fusion feature matrix: (154, 11712)
Acoustic audio slots: 3
Fusion audio slots: 3

Task-moment: AC
Transcript file: /home/projects/avera/medper/uvigo_voice/data/dicoperia/after_cough.csv
Acoustic feature matrix: (154, 6336)
Fusion feature matrix: (154, 11712)
Acoustic audio slots: 3
Fusion audio slots: 3

Task-moment: BC
Transcript file: /home/projects/avera/medper/uvigo_voice/data/dicoperia/before_cough.csv
Acoustic feature matrix: (154, 6336)
Fusion feature matrix: (154, 11712)
Acoustic audio slots: 3
Fusion audio slots: 3


,task_moment,representation,n_subjects,n_features,n_audio_slots
0,AA,acoustic,154,6336,3
1,AA,acoustic_wavlm_wav2vec,154,11712,3
2,BA,acoustic,154,6336,3
3,BA,acoustic_wavlm_wav2vec,154,11712,3
4,AC,acoustic,154,6336,3
5,AC,acoustic_wavlm_wav2vec,154,11712,3
6,BC,acoustic,154,6336,3
7,BC,acoustic_wavlm_wav2vec,154,11712,3


,task_moment,representation,n_features,n_raw_p_lt_005,n_fdr_p_lt_005,n_bonferroni_p_lt_005,min_raw_p,min_fdr_p,min_bonferroni_p,median_abs_effect_size,max_abs_effect_size
0,AA,acoustic,6336,1716,653,26,2.273053e-08,7.827993e-05,1.440206e-04,0.116327,0.524150
1,AA,acoustic_wavlm_wav2vec,11712,3026,1074,29,2.273053e-08,1.446993e-04,2.662199e-04,0.112585,0.524150
2,AC,acoustic,6336,2891,2093,374,1.201543e-11,1.094770e-08,7.612978e-08,0.171429,0.635714
3,AC,acoustic_wavlm_wav2vec,11712,5703,4499,649,9.243972e-14,1.082654e-09,1.082654e-09,0.179252,0.698639
4,BA,acoustic,6336,1709,863,188,5.099075e-10,7.522322e-07,3.230774e-06,0.110884,0.582823
5,BA,acoustic_wavlm_wav2vec,11712,3415,1682,233,1.323975e-10,1.291169e-06,1.550640e-06,0.119388,0.602381
6,BC,acoustic,6336,2290,1487,413,1.716547e-12,2.648538e-09,1.087604e-08,0.143878,0.661565
7,BC,acoustic_wavlm_wav2vec,11712,5066,3814,558,1.716547e-12,4.895782e-09,2.010420e-08,0.159864,0.661565


## Section 3 - Logistic Regression coefficient stability across CV folds

Evaluate coefficient stability by computing pairwise **Pearson correlations** between coefficient vectors from the five CV folds.

In [5]:
import pickle
from itertools import combinations
from scipy.stats import pearsonr

MODELS_BASE_DIR = Path("/home/temporal2/avera/coperia_raw/models_paper")

TASK_MODEL_DIRS = {
    "AA": MODELS_BASE_DIR / "after_a",
    "BA": MODELS_BASE_DIR / "before_a",
    "AC": MODELS_BASE_DIR / "after_cough",
    "BC": MODELS_BASE_DIR / "before_cough",
}

MODEL_CONFIGS = {
    "LR_acoustic_wav2vec_wavlm": (
        "aggregated_compare_2016_energy_compare_2016_voicing_compare_2016_rasta_"
        "compare_2016_basic_spectral_spafe_mfcc_ssl_wavlm_ssl_wav2vec_LogisticRegression_*.pkl"
    ),
    "LR_acoustic": (
        "aggregated_compare_2016_energy_compare_2016_voicing_compare_2016_rasta_"
        "compare_2016_basic_spectral_spafe_mfcc_LogisticRegression_*.pkl"
    ),
}


def load_pickle_model(path: Path):
    with open(path, "rb") as f:
        return pickle.load(f)


def get_lr_coef(model):
    if hasattr(model, "coef_"):
        return model.coef_.flatten()

    if hasattr(model, "model") and hasattr(model.model, "coef_"):
        return model.model.coef_.flatten()

    if hasattr(model, "classifier") and hasattr(model.classifier, "coef_"):
        return model.classifier.coef_.flatten()

    raise AttributeError(f"Could not find coef_ in object of type {type(model)}")


def find_fold_models(task_moment: str, configuration: str):
    task_dir = TASK_MODEL_DIRS[task_moment]

    pattern = MODEL_CONFIGS[configuration]
    model_paths = sorted(task_dir.glob(pattern))

    print(f"{task_moment} | {configuration}: found {len(model_paths)} models")

    return model_paths


def coefficient_stability_for_configuration(task_moment: str, configuration: str):
    model_paths = find_fold_models(task_moment, configuration)

    coefs = []
    fold_names = []

    for path in model_paths:
        model = load_pickle_model(path)
        coef = get_lr_coef(model)

        coefs.append(coef)
        fold_names.append(path.stem)

    coefs = np.asarray(coefs)

    if coefs.shape[0] < 2:
        print(f"Skipping {configuration}: fewer than 2 models found")
        return None, None

    rows = []

    for i, j in combinations(range(coefs.shape[0]), 2):
        r, p_value = pearsonr(coefs[i], coefs[j])

        rows.append({
            "task_moment": task_moment,
            "configuration": configuration,
            "fold_i": fold_names[i],
            "fold_j": fold_names[j],
            "pearson_r": r,
            "p_value": p_value,
        })

    pairwise_df = pd.DataFrame(rows)

    summary = {
        "task_moment": task_moment,
        "configuration": configuration,
        "n_folds": coefs.shape[0],
        "n_features": coefs.shape[1],
        "mean_pearson_r": pairwise_df["pearson_r"].mean(),
        "std_pearson_r": pairwise_df["pearson_r"].std(ddof=1),
        "min_pearson_r": pairwise_df["pearson_r"].min(),
        "max_pearson_r": pairwise_df["pearson_r"].max(),
    }

    return pairwise_df, summary


all_pairwise_corrs = []
stability_summaries = []

for task_moment in TASK_MODEL_DIRS.keys():
    for configuration in MODEL_CONFIGS.keys():

        pairwise_df, summary = coefficient_stability_for_configuration(
            task_moment,
            configuration,
        )

        if pairwise_df is not None:
            all_pairwise_corrs.append(pairwise_df)
            stability_summaries.append(summary)

coef_pairwise_correlations = pd.concat(all_pairwise_corrs, ignore_index=True)
coef_stability_summary = pd.DataFrame(stability_summaries)

coef_stability_summary = coef_stability_summary.sort_values(
    ["task_moment", "mean_pearson_r"],
    ascending=[True, False],
)

display(coef_stability_summary)
display(coef_pairwise_correlations)

TOP_K = 25

sign_consistency_rows = []

for task_moment in TASK_MODEL_DIRS.keys():
    for configuration in MODEL_CONFIGS.keys():

        model_paths = find_fold_models(task_moment, configuration)

        coefs = []
        for path in model_paths:
            model = load_pickle_model(path)
            coefs.append(get_lr_coef(model))

        coefs = np.asarray(coefs)

        if coefs.shape[0] < 2:
            continue

        mean_abs_coef = np.mean(np.abs(coefs), axis=0)
        top_idx = np.argsort(mean_abs_coef)[::-1][:TOP_K]

        for rank, feature_idx in enumerate(top_idx, start=1):
            signs = np.sign(coefs[:, feature_idx])

            n_positive = np.sum(signs > 0)
            n_negative = np.sum(signs < 0)
            n_zero = np.sum(signs == 0)

            sign_consistency = max(n_positive, n_negative, n_zero) / coefs.shape[0]

            sign_consistency_rows.append({
                "task_moment": task_moment,
                "configuration": configuration,
                "rank": rank,
                "feature_index": feature_idx,
                "mean_abs_coef": mean_abs_coef[feature_idx],
                "n_positive": n_positive,
                "n_negative": n_negative,
                "n_zero": n_zero,
                "sign_consistency": sign_consistency,
            })

coef_sign_consistency = pd.DataFrame(sign_consistency_rows)

coef_sign_consistency_summary = (
    coef_sign_consistency
    .groupby(["task_moment", "configuration"])
    .agg(
        top_k=("feature_index", "count"),
        mean_sign_consistency=("sign_consistency", "mean"),
        min_sign_consistency=("sign_consistency", "min"),
        max_sign_consistency=("sign_consistency", "max"),
    )
    .reset_index()
)

display(coef_sign_consistency_summary)
display(coef_sign_consistency)

AA | LR_acoustic_wav2vec_wavlm: found 5 models
AA | LR_acoustic: found 5 models
BA | LR_acoustic_wav2vec_wavlm: found 5 models
BA | LR_acoustic: found 5 models
AC | LR_acoustic_wav2vec_wavlm: found 5 models
AC | LR_acoustic: found 5 models
BC | LR_acoustic_wav2vec_wavlm: found 5 models
BC | LR_acoustic: found 5 models


,task_moment,configuration,n_folds,n_features,mean_pearson_r,std_pearson_r,min_pearson_r,max_pearson_r
0,AA,LR_acoustic_wav2vec_wavlm,5,11712,0.798964,0.023277,0.775124,0.843049
1,AA,LR_acoustic,5,6336,0.764798,0.028491,0.724683,0.813941
4,AC,LR_acoustic_wav2vec_wavlm,5,11712,0.772343,0.021066,0.739021,0.803617
5,AC,LR_acoustic,5,6336,0.736543,0.031116,0.695234,0.780751
2,BA,LR_acoustic_wav2vec_wavlm,5,11712,0.778216,0.032283,0.718730,0.825922
3,BA,LR_acoustic,5,6336,0.733502,0.036493,0.656746,0.774424
6,BC,LR_acoustic_wav2vec_wavlm,5,11712,0.763308,0.038317,0.706531,0.816172
7,BC,LR_acoustic,5,6336,0.754558,0.052684,0.689309,0.821037


,task_moment,configuration,fold_i,fold_j,pearson_r,p_value
0,AA,LR_acoustic_wav2vec_wavlm,aggregated_compare_2016_energy_compare_2016_vo...,aggregated_compare_2016_energy_compare_2016_vo...,0.788843,0.0
1,AA,LR_acoustic_wav2vec_wavlm,aggregated_compare_2016_energy_compare_2016_vo...,aggregated_compare_2016_energy_compare_2016_vo...,0.786769,0.0
2,AA,LR_acoustic_wav2vec_wavlm,aggregated_compare_2016_energy_compare_2016_vo...,aggregated_compare_2016_energy_compare_2016_vo...,0.778352,0.0
3,AA,LR_acoustic_wav2vec_wavlm,aggregated_compare_2016_energy_compare_2016_vo...,aggregated_compare_2016_energy_compare_2016_vo...,0.811525,0.0
4,AA,LR_acoustic_wav2vec_wavlm,aggregated_compare_2016_energy_compare_2016_vo...,aggregated_compare_2016_energy_compare_2016_vo...,0.775124,0.0
...,...,...,...,...,...,...
75,BC,LR_acoustic,aggregated_compare_2016_energy_compare_2016_vo...,aggregated_compare_2016_energy_compare_2016_vo...,0.689309,0.0
76,BC,LR_acoustic,aggregated_compare_2016_energy_compare_2016_vo...,aggregated_compare_2016_energy_compare_2016_vo...,0.820431,0.0
77,BC,LR_acoustic,aggregated_compare_2016_energy_compare_2016_vo...,aggregated_compare_2016_energy_compare_2016_vo...,0.695922,0.0
78,BC,LR_acoustic,aggregated_compare_2016_energy_compare_2016_vo...,aggregated_compare_2016_energy_compare_2016_vo...,0.821037,0.0


AA | LR_acoustic_wav2vec_wavlm: found 5 models
AA | LR_acoustic: found 5 models
BA | LR_acoustic_wav2vec_wavlm: found 5 models
BA | LR_acoustic: found 5 models
AC | LR_acoustic_wav2vec_wavlm: found 5 models
AC | LR_acoustic: found 5 models
BC | LR_acoustic_wav2vec_wavlm: found 5 models
BC | LR_acoustic: found 5 models


,task_moment,configuration,top_k,mean_sign_consistency,min_sign_consistency,max_sign_consistency
0,AA,LR_acoustic,25,1.000,1.0,1.0
1,AA,LR_acoustic_wav2vec_wavlm,25,1.000,1.0,1.0
2,AC,LR_acoustic,25,0.984,0.8,1.0
3,AC,LR_acoustic_wav2vec_wavlm,25,1.000,1.0,1.0
4,BA,LR_acoustic,25,1.000,1.0,1.0
5,BA,LR_acoustic_wav2vec_wavlm,25,1.000,1.0,1.0
6,BC,LR_acoustic,25,1.000,1.0,1.0
7,BC,LR_acoustic_wav2vec_wavlm,25,1.000,1.0,1.0


,task_moment,configuration,rank,feature_index,mean_abs_coef,n_positive,n_negative,n_zero,sign_consistency
0,AA,LR_acoustic_wav2vec_wavlm,1,8845,0.009171,0,5,0,1.0
1,AA,LR_acoustic_wav2vec_wavlm,2,1739,0.008987,5,0,0,1.0
2,AA,LR_acoustic_wav2vec_wavlm,3,8530,0.008935,0,5,0,1.0
3,AA,LR_acoustic_wav2vec_wavlm,4,7916,0.008814,0,5,0,1.0
4,AA,LR_acoustic_wav2vec_wavlm,5,6438,0.008731,0,5,0,1.0
...,...,...,...,...,...,...,...,...,...
195,BC,LR_acoustic,21,2339,0.016687,5,0,0,1.0
196,BC,LR_acoustic,22,1269,0.016596,5,0,0,1.0
197,BC,LR_acoustic,23,5634,0.016480,5,0,0,1.0
198,BC,LR_acoustic,24,1243,0.016431,5,0,0,1.0


## Section 4 — Coefficient variance across CV folds

Compute 95% confidence intervals across the five cross-validation folds for the average-probability metrics reported in the paper.

In [6]:
TOP_K = 25
CV_STABLE_THRESHOLD = 0.5
CV_STRICT_THRESHOLD = 0.3
EPSILON = 1e-8

coef_variability_rows = []
top_feature_variability_rows = []

for task_moment in TASK_MODEL_DIRS.keys():
    for configuration in MODEL_CONFIGS.keys():
        model_paths = find_fold_models(task_moment, configuration)

        coefs = []

        for path in model_paths:
            model = load_pickle_model(path)
            coefs.append(get_lr_coef(model))

        coefs = np.asarray(coefs)

        if coefs.shape[0] < 2:
            continue

        coef_mean = coefs.mean(axis=0)
        coef_std = coefs.std(axis=0, ddof=1)
        coef_cv = coef_std / (np.abs(coef_mean) + EPSILON)

        mean_abs_coef = np.mean(np.abs(coefs), axis=0)
        top_k_idx = np.argsort(mean_abs_coef)[::-1][:TOP_K]

        coef_variability_rows.append({
            "task_moment": task_moment,
            "configuration": configuration,
            "n_folds": coefs.shape[0],
            "n_features": coefs.shape[1],
            "mean_coef_std": coef_std.mean(),
            "median_coef_std": np.median(coef_std),
            "mean_coef_cv": coef_cv.mean(),
            "median_coef_cv": np.median(coef_cv),
            "stable_pct_cv_lt_0_5": np.mean(coef_cv < CV_STABLE_THRESHOLD) * 100,
            "strict_stable_pct_cv_lt_0_3": np.mean(coef_cv < CV_STRICT_THRESHOLD) * 100,
            f"top_{TOP_K}_mean_cv": coef_cv[top_k_idx].mean(),
            f"top_{TOP_K}_median_cv": np.median(coef_cv[top_k_idx]),
            f"top_{TOP_K}_stable_pct_cv_lt_0_5": np.mean(coef_cv[top_k_idx] < CV_STABLE_THRESHOLD) * 100,
            f"top_{TOP_K}_strict_stable_pct_cv_lt_0_3": np.mean(coef_cv[top_k_idx] < CV_STRICT_THRESHOLD) * 100,
        })

        for rank, feature_idx in enumerate(top_k_idx, start=1):
            top_feature_variability_rows.append({
                "task_moment": task_moment,
                "configuration": configuration,
                "rank": rank,
                "feature_index": feature_idx,
                "mean_abs_coef": mean_abs_coef[feature_idx],
                "mean_coef": coef_mean[feature_idx],
                "std_coef": coef_std[feature_idx],
                "cv_coef": coef_cv[feature_idx],
                "stable_cv_lt_0_5": coef_cv[feature_idx] < CV_STABLE_THRESHOLD,
                "strict_stable_cv_lt_0_3": coef_cv[feature_idx] < CV_STRICT_THRESHOLD,
            })

coef_variability_summary = pd.DataFrame(coef_variability_rows)
top_feature_variability = pd.DataFrame(top_feature_variability_rows)

display(coef_variability_summary)
display(top_feature_variability)

AA | LR_acoustic_wav2vec_wavlm: found 5 models
AA | LR_acoustic: found 5 models
BA | LR_acoustic_wav2vec_wavlm: found 5 models
BA | LR_acoustic: found 5 models
AC | LR_acoustic_wav2vec_wavlm: found 5 models
AC | LR_acoustic: found 5 models
BC | LR_acoustic_wav2vec_wavlm: found 5 models
BC | LR_acoustic: found 5 models


,task_moment,configuration,n_folds,n_features,mean_coef_std,median_coef_std,mean_coef_cv,median_coef_cv,stable_pct_cv_lt_0_5,strict_stable_pct_cv_lt_0_3,top_25_mean_cv,top_25_median_cv,top_25_stable_pct_cv_lt_0_5,top_25_strict_stable_pct_cv_lt_0_3
0,AA,LR_acoustic_wav2vec_wavlm,5,11712,0.000982,0.000909,9.388682,0.631622,40.975068,21.072404,0.128173,0.122569,100.0,100.0
1,AA,LR_acoustic,5,6336,0.001896,0.001720,10.271957,0.655982,37.452652,17.124369,0.161333,0.167378,100.0,100.0
2,BA,LR_acoustic_wav2vec_wavlm,5,11712,0.001058,0.000994,5.983357,0.726523,35.416667,17.400956,0.163259,0.152009,100.0,100.0
3,BA,LR_acoustic,5,6336,0.002116,0.001880,13.393392,0.861720,29.150884,12.500000,0.207242,0.186109,100.0,88.0
4,AC,LR_acoustic_wav2vec_wavlm,5,11712,0.001053,0.000965,4.823728,0.668844,37.687842,17.733948,0.184199,0.159012,96.0,92.0
5,AC,LR_acoustic,5,6336,0.002089,0.001800,3.448556,0.708254,36.158460,16.998106,0.294856,0.255655,92.0,72.0
6,BC,LR_acoustic_wav2vec_wavlm,5,11712,0.001171,0.001083,3.813586,0.739715,33.273566,14.002732,0.192055,0.206900,100.0,92.0
7,BC,LR_acoustic,5,6336,0.002102,0.001895,8.050111,0.719874,33.680556,13.731061,0.287448,0.283652,96.0,52.0


,task_moment,configuration,rank,feature_index,mean_abs_coef,mean_coef,std_coef,cv_coef,stable_cv_lt_0_5,strict_stable_cv_lt_0_3
0,AA,LR_acoustic_wav2vec_wavlm,1,8845,0.009171,-0.009171,0.001323,0.144204,True,True
1,AA,LR_acoustic_wav2vec_wavlm,2,1739,0.008987,0.008987,0.000905,0.100718,True,True
2,AA,LR_acoustic_wav2vec_wavlm,3,8530,0.008935,-0.008935,0.001291,0.144456,True,True
3,AA,LR_acoustic_wav2vec_wavlm,4,7916,0.008814,-0.008814,0.001152,0.130665,True,True
4,AA,LR_acoustic_wav2vec_wavlm,5,6438,0.008731,-0.008731,0.000744,0.085262,True,True
...,...,...,...,...,...,...,...,...,...,...
195,BC,LR_acoustic,21,2339,0.016687,0.016687,0.009091,0.544761,False,False
196,BC,LR_acoustic,22,1269,0.016596,0.016596,0.003041,0.183263,True,True
197,BC,LR_acoustic,23,5634,0.016480,0.016480,0.005535,0.335887,True,False
198,BC,LR_acoustic,24,1243,0.016431,0.016431,0.006417,0.390525,True,False


## SECTION 5 - Most important features overall:

In [20]:
from itertools import combinations

TOP_K = 25
SIGN_CONSISTENCY_THRESHOLD = 1.0  # Features with SAME sign in ALL folds (perfect consistency)
CV_STABLE_THRESHOLD = 0.5  # Coefficient variation threshold

# Rank features by mean absolute coefficient (overall importance across folds)
most_important_rows = []

for task_moment in TASK_MODEL_DIRS.keys():
    for configuration in MODEL_CONFIGS.keys():
        model_paths = find_fold_models(task_moment, configuration)

        coefs = []
        for path in model_paths:
            model = load_pickle_model(path)
            coefs.append(get_lr_coef(model))

        coefs = np.asarray(coefs)

        if coefs.shape[0] < 2:
            continue

        # Method 1: Rank by mean absolute coefficient (overall importance)
        mean_abs_coef = np.mean(np.abs(coefs), axis=0)
        overall_ranking = np.argsort(mean_abs_coef)[::-1]

        for rank, feature_idx in enumerate(overall_ranking[:TOP_K], start=1):
            most_important_rows.append({
                "task_moment": task_moment,
                "configuration": configuration,
                "rank": rank,
                "feature_index": feature_idx,
                "mean_abs_coef": mean_abs_coef[feature_idx],
            })

most_important_overall = pd.DataFrame(most_important_rows)

print("=" * 80)
print("FOUR-WAY COMPARISON: Discriminative Power vs. Stability vs. Magnitude")
print("=" * 80)
print("Section 2: Features with FDR-significant group differences (p_FDR < 0.05)")
print(f"Section 3: Features with sign_consistency == {SIGN_CONSISTENCY_THRESHOLD} (same sign across ALL folds)")
print(f"Section 4: Features with cv_coef < {CV_STABLE_THRESHOLD} (stable magnitude across folds)")
print("Section 5: Top-25 by Mean Absolute Coefficient (overall importance)\n")

AA | LR_acoustic_wav2vec_wavlm: found 5 models
AA | LR_acoustic: found 5 models
BA | LR_acoustic_wav2vec_wavlm: found 5 models
BA | LR_acoustic: found 5 models
AC | LR_acoustic_wav2vec_wavlm: found 5 models
AC | LR_acoustic: found 5 models
BC | LR_acoustic_wav2vec_wavlm: found 5 models
BC | LR_acoustic: found 5 models
FOUR-WAY COMPARISON: Discriminative Power vs. Stability vs. Magnitude
Section 2: Features with FDR-significant group differences (p_FDR < 0.05)
Section 3: Features with sign_consistency == 1.0 (same sign across ALL folds)
Section 4: Features with cv_coef < 0.5 (stable magnitude across folds)
Section 5: Top-25 by Mean Absolute Coefficient (overall importance)



In [21]:

# Build Section 2 features (Mann-Whitney significant features)
section2_features_by_config = {}

for task_moment in TASK_MOMENT_FILES.keys():
    for representation in ["acoustic", "acoustic_wavlm_wav2vec"]:
        config_key = (task_moment, representation)
        
        # Get significant features from Mann-Whitney (FDR corrected, p < 0.05)
        sig_features = mw_results[
            (mw_results["task_moment"] == task_moment) &
            (mw_results["representation"] == representation) &
            (mw_results["significant_fdr"] == True)
        ]["feature_index"].values
        
        section2_features_by_config[config_key] = set(sig_features)

comparison_four_way = []

for task_moment in TASK_MODEL_DIRS.keys():
    for configuration in MODEL_CONFIGS.keys():
        # Determine which representation to use for Section 2
        if configuration == "LR_acoustic":
            representation = "acoustic"
        else:  # LR_acoustic_wav2vec_wavlm
            representation = "acoustic_wavlm_wav2vec"
        
        config_key_section2 = (task_moment, representation)
        
        # Get features from all four sections
        section2_features = section2_features_by_config.get(config_key_section2, set())
        
        # Get Section 3 features (Sign Consistency == 1.0, same sign in ALL folds)
        section3_features = set(
            coef_sign_consistency[
                (coef_sign_consistency["task_moment"] == task_moment) &
                (coef_sign_consistency["configuration"] == configuration) &
                (coef_sign_consistency["sign_consistency"] == SIGN_CONSISTENCY_THRESHOLD)
            ]["feature_index"].values
        )
        
        # Get Section 4 features (CV < 0.5, stable magnitude)
        section4_features = set(
            top_feature_variability[
                (top_feature_variability["task_moment"] == task_moment) &
                (top_feature_variability["configuration"] == configuration) &
                (top_feature_variability["cv_coef"] < CV_STABLE_THRESHOLD)
            ]["feature_index"].values
        )
        
        # Get top-25 from Section 5 (Mean Absolute Coefficient)
        section5_features = set(
            most_important_overall[
                (most_important_overall["task_moment"] == task_moment) &
                (most_important_overall["configuration"] == configuration) &
                (most_important_overall["rank"] <= TOP_K)
            ]["feature_index"].values
        )
        
        all_four = section2_features & section3_features & section4_features & section5_features
        only_2 = section2_features - section3_features - section4_features - section5_features
        only_3 = section3_features - section2_features - section4_features - section5_features
        only_4 = section4_features - section2_features - section3_features - section5_features
        only_5 = section5_features - section2_features - section3_features - section4_features
        
        comparison_four_way.append({
            "task_moment": task_moment,
            "configuration": configuration,
            "n_mw_significant": len(section2_features),
            "n_all_four": len(all_four),
            "n_mw_sign_coef_var": len((section2_features & section3_features & section4_features) - section5_features),
            "n_mw_sign_coef": len((section2_features & section3_features & section5_features) - section4_features),
            "n_mw_coef_var": len((section2_features & section4_features & section5_features) - section3_features),
            "n_sign_coef_var": len((section3_features & section4_features & section5_features) - section2_features),
            "n_only_mw": len(only_2),
            "n_only_sign": len(only_3),
            "n_only_var": len(only_4),
            "n_only_coef": len(only_5),
        })

four_way_df = pd.DataFrame(comparison_four_way)

print("\nFour-Way Overlap Summary (with updated criteria):")
print("-" * 80)
print("Section 2: Mann-Whitney FDR-significant (p_FDR < 0.05)")
print("Section 3: Sign consistency == 1.0 (SAME sign in ALL folds)")
print("Section 4: CV < 0.5 (stable magnitude)")
print("Section 5: Top-25 by mean absolute coefficient")
print()
display(four_way_df)


Four-Way Overlap Summary (with updated criteria):
--------------------------------------------------------------------------------
Section 2: Mann-Whitney FDR-significant (p_FDR < 0.05)
Section 3: Sign consistency == 1.0 (SAME sign in ALL folds)
Section 4: CV < 0.5 (stable magnitude)
Section 5: Top-25 by mean absolute coefficient



,task_moment,configuration,n_mw_significant,n_all_four,n_mw_sign_coef_var,n_mw_sign_coef,n_mw_coef_var,n_sign_coef_var,n_only_mw,n_only_sign,n_only_var,n_only_coef
0,AA,LR_acoustic_wav2vec_wavlm,1074,15,0,0,0,10,1059,0,0,0
1,AA,LR_acoustic,653,22,0,0,0,3,631,0,0,0
2,BA,LR_acoustic_wav2vec_wavlm,1682,22,0,0,0,3,1660,0,0,0
3,BA,LR_acoustic,863,21,0,0,0,4,842,0,0,0
4,AC,LR_acoustic_wav2vec_wavlm,4499,18,0,0,0,6,4481,0,0,0
5,AC,LR_acoustic,2093,13,0,0,0,10,2079,0,0,1
6,BC,LR_acoustic_wav2vec_wavlm,3814,13,0,0,0,12,3801,0,0,0
7,BC,LR_acoustic,1487,8,0,0,0,16,1479,0,0,0


## Section 6 - Feature Index Mapping: Top 25 Features Identification

Identify which acoustic and SSL features correspond to the top 25 feature indices from each model.
For acoustic features, pinpoint the specific feature type and aggregation statistic.
For SSL features, identify the audio slot and feature dimension.


In [15]:
def format_feature_info(feature_info, global_idx):
    """Format feature information as a readable string"""
    if feature_info is None:
        return "Unknown feature"
    
    audio_slot = feature_info.get('audio_slot', 'N/A')
    feature_type = feature_info.get('feature_type', 'N/A')
    
    # Case 1: Acoustic within SSL model
    if feature_type == 'acoustic':
        detail = feature_info['detail']
        if detail is None:
            return f"Acoustic (Audio {audio_slot}): Unknown"
        feat_type = detail['feature_type'].upper()
        raw_id = detail['raw_feature_id']
        stat = detail['aggregation_stat']
        return f"Acoustic (Audio {audio_slot}): {feat_type} feat#{raw_id} [{stat}]"
    
    # Case 2: Pure acoustic model (acoustic-only)
    elif feature_type in ['energy', 'voicing', 'rasta', 'basic_spectral', 'mfcc']:
        feat_type = feature_type.upper()
        raw_id = feature_info.get('raw_feature_id', 'N/A')
        stat = feature_info.get('aggregation_stat', 'N/A')
        return f"Acoustic (Audio {audio_slot}): {feat_type} feat#{raw_id} [{stat}]"
    
    # Case 3: WavLM SSL feature
    elif feature_type == 'ssl_wavlm':
        wavlm_dim = feature_info.get('wavlm_dimension', 'N/A')
        return f"WavLM (Audio {audio_slot}): dim {wavlm_dim}"
    
    # Case 4: Wav2Vec SSL feature
    elif feature_type == 'ssl_wav2vec':
        wav2vec_dim = feature_info.get('wav2vec_dimension', 'N/A')
        return f"Wav2Vec (Audio {audio_slot}): dim {wav2vec_dim}"
    
    else:
        return f"Unknown feature at index {global_idx}"


print("Feature mapping utilities loaded successfully!")

Feature mapping utilities loaded successfully!


In [17]:

# Feature decoding utilities
ACOUSTIC_FEATURE_BREAKDOWN = {
    'energy': {'raw_features': 12, 'index_range': (0, 132)},
    'voicing': {'raw_features': 18, 'index_range': (132, 330)},
    'rasta': {'raw_features': 78, 'index_range': (330, 1188)},
    'basic_spectral': {'raw_features': 45, 'index_range': (1188, 1683)},
    'mfcc': {'raw_features': 39, 'index_range': (1683, 2112)},
}
AGGREGATION_STATS = ['mean', 'std', 'min', 'max', 'entropy', 'skew', 'kurtosis', 'q1', 'q2', 'q3', 'q4']
FEATURES_PER_AUDIO_ACOUSTIC = 2112
SSL_WAVLM_DIM = 768
SSL_WAV2VEC_DIM = 1024
FEATURES_PER_AUDIO_SSL_COMBINED = 3904
N_AUDIOS = 3

def decode_acoustic_feature_index(global_idx):
    """Decode a global feature index for acoustic-only model (0-6335)"""
    if global_idx >= N_AUDIOS * FEATURES_PER_AUDIO_ACOUSTIC:
        return None
    
    audio_slot = global_idx // FEATURES_PER_AUDIO_ACOUSTIC
    local_idx = global_idx % FEATURES_PER_AUDIO_ACOUSTIC
    
    for feat_type, info in ACOUSTIC_FEATURE_BREAKDOWN.items():
        start, end = info['index_range']
        if start <= local_idx < end:
            raw_features = info['raw_features']
            within_range = local_idx - start
            raw_feature_id = within_range // len(AGGREGATION_STATS)
            agg_stat_idx = within_range % len(AGGREGATION_STATS)
            
            return {
                'feature_type': feat_type,
                'audio_slot': audio_slot,
                'raw_feature_id': raw_feature_id,
                'aggregation_stat': AGGREGATION_STATS[agg_stat_idx],
            }
    
    return None

def decode_ssl_feature_index(global_idx, is_acoustic_only=False):
    """Decode a global feature index for SSL model or acoustic-only model"""
    if is_acoustic_only:
        if global_idx >= N_AUDIOS * FEATURES_PER_AUDIO_ACOUSTIC:
            return None
        
        audio_slot = global_idx // FEATURES_PER_AUDIO_ACOUSTIC
        local_idx = global_idx % FEATURES_PER_AUDIO_ACOUSTIC
        
        for feat_type, info in ACOUSTIC_FEATURE_BREAKDOWN.items():
            start, end = info['index_range']
            if start <= local_idx < end:
                raw_features = info['raw_features']
                within_range = local_idx - start
                raw_feature_id = within_range // len(AGGREGATION_STATS)
                agg_stat_idx = within_range % len(AGGREGATION_STATS)
                
                return {
                    'feature_type': feat_type,
                    'audio_slot': audio_slot,
                    'raw_feature_id': raw_feature_id,
                    'aggregation_stat': AGGREGATION_STATS[agg_stat_idx],
                }
        return None
    
    else:
        if global_idx >= N_AUDIOS * FEATURES_PER_AUDIO_SSL_COMBINED:
            return None
        
        audio_slot = global_idx // FEATURES_PER_AUDIO_SSL_COMBINED
        local_idx = global_idx % FEATURES_PER_AUDIO_SSL_COMBINED
        
        if local_idx < FEATURES_PER_AUDIO_ACOUSTIC:
            for feat_type, info in ACOUSTIC_FEATURE_BREAKDOWN.items():
                start, end = info['index_range']
                if start <= local_idx < end:
                    within_range = local_idx - start
                    raw_feature_id = within_range // len(AGGREGATION_STATS)
                    agg_stat_idx = within_range % len(AGGREGATION_STATS)
                    
                    return {
                        'feature_type': 'acoustic',
                        'audio_slot': audio_slot,
                        'detail': {
                            'feature_type': feat_type,
                            'raw_feature_id': raw_feature_id,
                            'aggregation_stat': AGGREGATION_STATS[agg_stat_idx],
                        }
                    }
            return None
        
        elif local_idx < FEATURES_PER_AUDIO_ACOUSTIC + SSL_WAVLM_DIM:
            wavlm_dim = local_idx - FEATURES_PER_AUDIO_ACOUSTIC
            return {
                'feature_type': 'ssl_wavlm',
                'audio_slot': audio_slot,
                'wavlm_dimension': wavlm_dim,
            }
        
        elif local_idx < FEATURES_PER_AUDIO_ACOUSTIC + SSL_WAVLM_DIM + SSL_WAV2VEC_DIM:
            wav2vec_dim = local_idx - FEATURES_PER_AUDIO_ACOUSTIC - SSL_WAVLM_DIM
            return {
                'feature_type': 'ssl_wav2vec',
                'audio_slot': audio_slot,
                'wav2vec_dimension': wav2vec_dim,
            }
        
        return None


In [18]:
print("="*100)
print("TOP 25 FEATURES IDENTIFICATION BY MODEL")
print("="*100)

# Get top 25 features for each configuration from Section 5 (most_important_overall)
for task_moment in TASK_MODEL_DIRS.keys():
    print(f"\n{'='*100}")
    print(f"TASK-MOMENT: {task_moment}")
    print(f"{'='*100}")
    
    for configuration in MODEL_CONFIGS.keys():
        is_acoustic_only = (configuration == "LR_acoustic")
        
        # Get top 25 features for this configuration
        top_features = most_important_overall[
            (most_important_overall["task_moment"] == task_moment) &
            (most_important_overall["configuration"] == configuration) &
            (most_important_overall["rank"] <= 25)
        ].copy()
        
        if len(top_features) == 0:
            print(f"\n{configuration}: No top-25 features found")
            continue
        
        print(f"\n{configuration}:")
        print(f"  Model type: {'Acoustic-only (6336 features)' if is_acoustic_only else 'Acoustic+SSL (11712 features)'}")
        print(f"\n  {'Rank':<5} {'Feature Index':<15} {'Feature Description':<60} {'Mean |Coef|':<12}")
        print(f"  {'-'*92}")
        
        # Decode each feature index
        for _, row in top_features.iterrows():
            global_idx = int(row['feature_index'])
            rank = int(row['rank'])
            mean_coef = row['mean_abs_coef']
            
            if is_acoustic_only:
                feature_info = decode_acoustic_feature_index(global_idx)
            else:
                feature_info = decode_ssl_feature_index(global_idx, is_acoustic_only=False)
            
            feature_desc = format_feature_info(feature_info, global_idx)
            
            print(f"  {rank:<5} {global_idx:<15} {feature_desc:<60} {mean_coef:<12.4f}")
        
        # Summary statistics
        print(f"\n  Summary for {configuration}:")
        if is_acoustic_only:
            # Count acoustic features by type
            feature_types = {}
            audio_slots = {}
            agg_stats = {}
            
            for idx in top_features['feature_index'].values:
                info = decode_acoustic_feature_index(int(idx))
                if info:
                    ftype = info['feature_type']
                    audio = info['audio_slot']
                    stat = info['aggregation_stat']
                    
                    feature_types[ftype] = feature_types.get(ftype, 0) + 1
                    audio_slots[audio] = audio_slots.get(audio, 0) + 1
                    agg_stats[stat] = agg_stats.get(stat, 0) + 1
            
            print(f"    - By feature type: {dict(sorted(feature_types.items()))}")
            print(f"    - By audio slot: {dict(sorted(audio_slots.items()))}")
            print(f"    - By aggregation stat: {dict(sorted(agg_stats.items(), key=lambda x: x[1], reverse=True)[:5])}...")
        
        else:
            # Count features by type (acoustic vs SSL)
            feature_type_counts = {'acoustic': 0, 'ssl_wavlm': 0, 'ssl_wav2vec': 0}
            audio_slots = {}
            
            for idx in top_features['feature_index'].values:
                info = decode_ssl_feature_index(int(idx), is_acoustic_only=False)
                if info:
                    ftype = info['feature_type']
                    audio = info['audio_slot']
                    
                    if ftype == 'acoustic':
                        feature_type_counts['acoustic'] += 1
                    elif ftype.startswith('ssl_wavlm'):
                        feature_type_counts['ssl_wavlm'] += 1
                    elif ftype.startswith('ssl_wav2vec'):
                        feature_type_counts['ssl_wav2vec'] += 1
                    
                    audio_slots[audio] = audio_slots.get(audio, 0) + 1
            
            total = sum(feature_type_counts.values())
            print(f"    - Acoustic features: {feature_type_counts['acoustic']}/{total} ({100*feature_type_counts['acoustic']/total:.1f}%)")
            print(f"    - WavLM features: {feature_type_counts['ssl_wavlm']}/{total} ({100*feature_type_counts['ssl_wavlm']/total:.1f}%)")
            print(f"    - Wav2Vec features: {feature_type_counts['ssl_wav2vec']}/{total} ({100*feature_type_counts['ssl_wav2vec']/total:.1f}%)")
            print(f"    - By audio slot: {dict(sorted(audio_slots.items()))}")
            
            # If acoustic features are present, show their breakdown
            if feature_type_counts['acoustic'] > 0:
                feature_types = {}
                agg_stats = {}
                for idx in top_features['feature_index'].values:
                    info = decode_ssl_feature_index(int(idx), is_acoustic_only=False)
                    if info and info['feature_type'] == 'acoustic':
                        detail = info['detail']
                        if detail:
                            ftype = detail['feature_type']
                            stat = detail['aggregation_stat']
                            feature_types[ftype] = feature_types.get(ftype, 0) + 1
                            agg_stats[stat] = agg_stats.get(stat, 0) + 1
                
                if feature_types:
                    print(f"    - Acoustic by type: {dict(sorted(feature_types.items()))}")
                if agg_stats:
                    top_stats = dict(sorted(agg_stats.items(), key=lambda x: x[1], reverse=True)[:5])
                    print(f"    - Acoustic agg stats (top): {top_stats}")

print("\n" + "="*100)
print("END OF FEATURE IDENTIFICATION")
print("="*100)


TOP 25 FEATURES IDENTIFICATION BY MODEL

TASK-MOMENT: AA

LR_acoustic_wav2vec_wavlm:
  Model type: Acoustic+SSL (11712 features)

  Rank  Feature Index   Feature Description                                          Mean |Coef| 
  --------------------------------------------------------------------------------------------
  1     8845            Acoustic (Audio 2): RASTA feat#64 [max]                      0.0092      
  2     1739            Acoustic (Audio 0): MFCC feat#5 [std]                        0.0090      
  3     8530            Acoustic (Audio 2): RASTA feat#35 [q1]                       0.0089      
  4     7916            Acoustic (Audio 2): ENERGY feat#9 [q3]                       0.0088      
  5     6438            WavLM (Audio 1): dim 422                                     0.0087      
  6     6131            WavLM (Audio 1): dim 115                                     0.0083      
  7     5629            Acoustic (Audio 1): MFCC feat#3 [q3]                         0.00

## Section 6a - Export Feature Identification Results to CSV

Save the detailed feature identification results to CSV files for further analysis and reporting.


In [19]:
# ============================================================================
# Export Feature Identification Results to CSV
# ============================================================================

output_dir = Path('./feature_identification_results')
output_dir.mkdir(exist_ok=True)

# Create comprehensive results dataframe
feature_results_rows = []

for task_moment in TASK_MODEL_DIRS.keys():
    for configuration in MODEL_CONFIGS.keys():
        is_acoustic_only = (configuration == "LR_acoustic")
        
        # Get top 25 features for this configuration
        top_features = most_important_overall[
            (most_important_overall["task_moment"] == task_moment) &
            (most_important_overall["configuration"] == configuration) &
            (most_important_overall["rank"] <= 25)
        ].copy()
        
        if len(top_features) == 0:
            continue
        
        # Decode each feature
        for _, row in top_features.iterrows():
            global_idx = int(row['feature_index'])
            rank = int(row['rank'])
            mean_coef = row['mean_abs_coef']
            
            # Use decode_ssl_feature_index for both acoustic and SSL models
            # For acoustic-only: 3 audios × 2112 features = 0-6335
            # For SSL: 3 audios × 3904 features = 0-11711
            feature_info = decode_ssl_feature_index(global_idx, is_acoustic_only=is_acoustic_only)
            
            feature_desc = format_feature_info(feature_info, global_idx)
            
            # Determine if feature is in all-four agreement
            if configuration == "LR_acoustic":
                representation = "acoustic"
            else:
                representation = "acoustic_wavlm_wav2vec"
            
            config_key_section2 = (task_moment, representation)
            section2_features = section2_features_by_config.get(config_key_section2, set())
            
            section3_features = set(
                coef_sign_consistency[
                    (coef_sign_consistency["task_moment"] == task_moment) &
                    (coef_sign_consistency["configuration"] == configuration) &
                    (coef_sign_consistency["rank"] <= TOP_K)
                ]["feature_index"].values
            )
            
            section4_data_filtered = top_feature_variability[
                (top_feature_variability["task_moment"] == task_moment) &
                (top_feature_variability["configuration"] == configuration)
            ].copy()
            
            if len(section4_data_filtered) > 0:
                section4_data_filtered = section4_data_filtered.sort_values("cv_coef").head(TOP_K)
                section4_features = set(section4_data_filtered["feature_index"].values)
            else:
                section4_features = set()
            
            section5_features = set(
                most_important_overall[
                    (most_important_overall["task_moment"] == task_moment) &
                    (most_important_overall["configuration"] == configuration) &
                    (most_important_overall["rank"] <= TOP_K)
                ]["feature_index"].values
            )
            
            is_all_four = (global_idx in section2_features and 
                          global_idx in section3_features and 
                          global_idx in section4_features and 
                          global_idx in section5_features)
            
            is_mw_significant = global_idx in section2_features
            
            feature_results_rows.append({
                'task_moment': task_moment,
                'configuration': configuration,
                'rank': rank,
                'feature_index': global_idx,
                'feature_description': feature_desc,
                'mean_abs_coefficient': mean_coef,
                'is_all_four_methods': is_all_four,
                'is_mann_whitney_significant': is_mw_significant,
            })

feature_results_df = pd.DataFrame(feature_results_rows)

# Save to CSV
csv_path = output_dir / 'top_25_features_identification.csv'
feature_results_df.to_csv(csv_path, sep=';', decimal=',', index=False)

print(f"✓ Feature identification results saved to: {csv_path}")
print(f"  Total rows: {len(feature_results_df)}")
print(f"  Columns: {', '.join(feature_results_df.columns.tolist())}")

# Summary statistics
print(f"\nSummary of exported features:")
summary_stats = feature_results_df.groupby(['task_moment', 'configuration']).agg({
    'is_all_four_methods': 'sum',
    'is_mann_whitney_significant': 'sum',
}).rename(columns={
    'is_all_four_methods': 'n_all_four',
    'is_mann_whitney_significant': 'n_mw_significant'
})
print(summary_stats)

# Also export the all-four features and significant features separately for easier filtering
all_four_features = feature_results_df[feature_results_df['is_all_four_methods'] == True].copy()
all_four_path = output_dir / 'all_four_methods_features.csv'
all_four_features.to_csv(all_four_path, sep=';', decimal=',', index=False)
print(f"\n✓ All-four methods features saved to: {all_four_path}")

mw_significant_features = feature_results_df[feature_results_df['is_mann_whitney_significant'] == True].copy()
mw_sig_path = output_dir / 'mann_whitney_significant_features.csv'
mw_significant_features.to_csv(mw_sig_path, sep=';', decimal=',', index=False)
print(f"✓ Mann-Whitney significant features saved to: {mw_sig_path}")

display(feature_results_df.head(10))

✓ Feature identification results saved to: feature_identification_results/top_25_features_identification.csv
  Total rows: 200
  Columns: task_moment, configuration, rank, feature_index, feature_description, mean_abs_coefficient, is_all_four_methods, is_mann_whitney_significant

Summary of exported features:
                                       n_all_four  n_mw_significant
task_moment configuration                                          
AA          LR_acoustic                        22                22
            LR_acoustic_wav2vec_wavlm          15                15
AC          LR_acoustic                        14                14
            LR_acoustic_wav2vec_wavlm          18                18
BA          LR_acoustic                        21                21
            LR_acoustic_wav2vec_wavlm          22                22
BC          LR_acoustic                         8                 8
            LR_acoustic_wav2vec_wavlm          13                13

✓ All-fou

,task_moment,configuration,rank,feature_index,feature_description,mean_abs_coefficient,is_all_four_methods,is_mann_whitney_significant
0,AA,LR_acoustic_wav2vec_wavlm,1,8845,Acoustic (Audio 2): RASTA feat#64 [max],0.009171,False,False
1,AA,LR_acoustic_wav2vec_wavlm,2,1739,Acoustic (Audio 0): MFCC feat#5 [std],0.008987,True,True
2,AA,LR_acoustic_wav2vec_wavlm,3,8530,Acoustic (Audio 2): RASTA feat#35 [q1],0.008935,True,True
3,AA,LR_acoustic_wav2vec_wavlm,4,7916,Acoustic (Audio 2): ENERGY feat#9 [q3],0.008814,True,True
4,AA,LR_acoustic_wav2vec_wavlm,5,6438,WavLM (Audio 1): dim 422,0.008731,True,True
5,AA,LR_acoustic_wav2vec_wavlm,6,6131,WavLM (Audio 1): dim 115,0.008290,True,True
6,AA,LR_acoustic_wav2vec_wavlm,7,5629,Acoustic (Audio 1): MFCC feat#3 [q3],0.008220,True,True
7,AA,LR_acoustic_wav2vec_wavlm,8,5948,Acoustic (Audio 1): MFCC feat#32 [q3],0.008164,False,False
8,AA,LR_acoustic_wav2vec_wavlm,9,6149,WavLM (Audio 1): dim 133,0.008129,False,False
9,AA,LR_acoustic_wav2vec_wavlm,10,8235,Acoustic (Audio 2): RASTA feat#8 [q3],0.008086,True,True
